# 6th attempt - RCNN - 5

In [16]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import tensorflow as tf

In [17]:
SIZE = 5
AMOUNT_BOARDS = 10000

In [18]:
gen = 2
name_df = f'{PATH_DF}\\{SIZE}-{AMOUNT_BOARDS}\\{SIZE}size_{AMOUNT_BOARDS}boards_{gen}gen_reverse'
reverse_df = pd.read_pickle(f'{name_df}.pkl')

In [19]:
new_columns = [f'Col_{i}' for i in range(gen*SIZE*SIZE)]
reverse_df_sort = reverse_df.sort_values(by = new_columns).reset_index(drop=True)
for i in reverse_df_sort.columns:
    reverse_df_sort[i] = reverse_df_sort[i].astype(int)

In [20]:
print("reverse df:", len(reverse_df))
print("reverse df sort:",len(reverse_df_sort))

reverse df: 29760
reverse df sort: 29760


In [21]:
# Step 1: Prepare Data
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
name_col = 'Col_' + str(amount_features + 1)  # Target: the first pixel in the board
target = reverse_df_sort[name_col]

# Step 2: Split Data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1, random_state=365)

print("len x train: ", len(X_train))
print("len x test: ",len(X_test))
print("len y train: ",len(y_train))
print("len y test: ",len(y_test))

len x train:  26784
len x test:  2976
len y train:  26784
len y test:  2976


In [22]:
X_train_array = X_train.to_numpy()
y_train_array = y_train.to_numpy()
X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
y_train_array = y_train_array.reshape((y_train.shape[0],1))

In [23]:
X_test_array = X_test.to_numpy()
X_test_array = X_test_array.reshape((X_test.shape[0],SIZE,SIZE,1))
y_test_array = y_test.to_numpy()
y_test_array = y_test_array.reshape((y_test.shape[0],1))

In [24]:
def build_RCNN5_sidmoind(gen_data, x_train, y_train, size, batch_size, epochs):
    # --- PREPROCESSING ---

    num_samples = x_train.shape[0] - gen_data

    X_train = np.zeros((num_samples, gen_data, size, size, 1), dtype='float32')
    Y_train = np.zeros((num_samples, 1), dtype='float32')  # רק תא אחד

    for i in range(num_samples):
        X_train[i] = x_train[i:i+gen_data]   # רצף של gen_data לוחות
        Y_train[i] = y_train[i]              # הפלט: תא אחד (0/1)

    print("X_train shape:", X_train.shape)  # (num_samples, gen_data, SIZE, SIZE, 1)
    print("y_train shape:", Y_train.shape)  # (num_samples, 1)

    # --- MODEL ---
    model = tf.keras.Sequential([
        tf.keras.layers.ConvLSTM2D(
            filters=32,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=True,
            input_shape=(gen_data, SIZE, SIZE, 1)
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.ConvLSTM2D(
            filters=64,
            kernel_size=(3,3),
            activation='relu',
            padding='same',
            return_sequences=False
        ),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')  # ← פלט יחיד בינארי
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    # אימון
    history = model.fit(
        X_train, Y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        shuffle=True
    )
    
    return model

In [25]:
gen_data = gen-1
model = build_RCNN5_sidmoind(gen_data, X_train_array, y_train_array, SIZE, 32, 3)

X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_4 (ConvLSTM2D)      │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_5 (ConvLSTM2D)      │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,497 (1.38 MB)

 Trainable params: 362,305 (1.38 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 180s 171ms/step - accuracy: 0.6692 - loss: 0.6237 - val_accuracy: 0.6823 - val_loss: 0.6152
Epoch 2/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 56s 83ms/step - accuracy: 0.6930 - loss: 0.5883 - val_accuracy: 0.6946 - val_loss: 0.5863
Epoch 3/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 28s 39ms/step - accuracy: 0.7109 - loss: 0.5654 - val_accuracy: 0.6974 - val_loss: 0.5842


In [26]:
X_test = X_test_array.reshape((-1, gen_data, SIZE, SIZE, 1)).astype('float32')
y_test = y_test_array.astype('float32')

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6851 - loss: 0.6021
Test accuracy: 0.685


In [27]:
import numpy as np
from sklearn.metrics import confusion_matrix

def evaluate_model(model, X_test_array, y_test_array, gen_data, SIZE):
    """
    מעריך את ביצועי המודל ומציג טבלת Confusion Matrix ומדדים מסוכמים.
    """
    # עיבוד הנתונים
    X_test = X_test_array.reshape((-1, gen_data, SIZE, SIZE, 1)).astype('float32')
    y_test = y_test_array.reshape((-1, 1)).astype('float32')

    # חיזוי
    y_pred = model.predict(X_test)
    y_pred_binary = (y_pred > 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_binary)
    tn, fp, fn, tp = cm.ravel()

    # חישוב מדדים
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    acc = (tp + tn) / (tp + tn + fp + fn)

    # --- הדפסה בטבלה ---
    print("\n===== Evaluation Results =====")
    print("┌──────────────┬────────────┬────────────┐")
    print(f"│              │ Pred=Alive │ Pred=Dead  │")
    print("├──────────────┼────────────┼────────────┤")
    print(f"│ True=Alive   │ {tp:10d} │ {fn:10d} │")
    print(f"│ True=Dead    │ {fp:10d} │ {tn:10d} │")
    print("└──────────────┴────────────┴────────────┘")

    print("\n--- Performance Metrics ---")
    print(f"{'Accuracy':<12}: {acc:.3f}")
    print(f"{'Precision':<12}: {precision:.3f}")
    print(f"{'Recall':<12}: {recall:.3f}")
    print(f"{'F1-score':<12}: {f1:.3f}")


In [28]:
# יצירת סט בדיקה
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test, 1), dtype='float32')

for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data]
    y_test[i] = y_test_array[i]

# הפעלת פונקציית ההערכה
results = evaluate_model(model, X_test, y_test, gen_data, SIZE)


93/93 ━━━━━━━━━━━━━━━━━━━━ 45s 362ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        292 │        761 │
│ True=Dead    │        177 │       1745 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.685
Precision   : 0.623
Recall      : 0.277
F1-score    : 0.384


In [29]:
# Step 1: Prepare Data
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
name_col = 'Col_' + str(amount_features + 1)  # Target: the first pixel in the board
target = reverse_df_sort[name_col]

# Step 2: Split Data
X_train_val, X_test, y_train_val, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=365)

print("len x train: ", len(X_train))
print("len x test: ",len(X_test))
print("len y train: ",len(y_train))
print("len y test: ",len(y_test))

len x train:  24105
len x test:  2976
len y train:  24105
len y test:  2976


In [30]:
amount_features = len(reverse_df_sort.columns) - SIZE*SIZE #the previous boards
features = reverse_df_sort.iloc[:, :amount_features]
for i in range(SIZE*SIZE): # to any pixel in the expected board
    name_col = 'Col_' + str(i+amount_features)
    target = reverse_df_sort[name_col]
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.1, random_state=365)
    
    X_train_array = X_train.to_numpy()
    y_train_array = y_train.to_numpy()
    X_train_array = X_train_array.reshape((X_train.shape[0],SIZE,SIZE,1))
    y_train_array = y_train_array.reshape((y_train.shape[0],1))
    
    model = build_RCNN5_sidmoind(gen_data, X_train_array, y_train_array, SIZE, 32, 3)
    name_file = f"{PATH_MODELS}\\model6_RCNN_sigmoid\\{SIZE}\\size_{SIZE}_pixel_{str(i+1)}.pkl"
    with open(name_file, 'wb') as file:
        pickle.dump(model, file)
    print(i)

X_train shape: (26783, 1, 5, 5, 1)
y_train shape: (26783, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_6 (ConvLSTM2D)      │ (None, 1, 5, 5, 32)    │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 1, 5, 5, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_7 (ConvLSTM2D)      │ (None, 5, 5, 64)       │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 5, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,497 (1.38 MB)

 Trainable params: 362,305 (1.38 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 275s 170ms/step - accuracy: 0.6685 - loss: 0.6221 - val_accuracy: 0.6784 - val_loss: 0.6040
Epoch 2/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 64s 84ms/step - accuracy: 0.6985 - loss: 0.5824 - val_accuracy: 0.6972 - val_loss: 0.5831
Epoch 3/3
670/670 ━━━━━━━━━━━━━━━━━━━━ 57s 50ms/step - accuracy: 0.7112 - loss: 0.5646 - val_accuracy: 0.6982 - val_loss: 0.5873


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\GameOfLifeFiles\\models\\\\model6_RCNN_sigmoid\\5\\size_5_pixel_1.pkl'